In [1]:
import fitz  # PyMuPDF library
import os

# 1. Define the correct path to your PDF file inside the CVs folder
pdf_path = "CVs/ResumeCyberSecurityAnalystBasic.pdf" 

def extract_text_from_pdf(file_path):
    """Function to open a PDF file and extract all its text content."""
    try:
        # Open the PDF document
        doc = fitz.open(file_path)
        text = ""
        
        # Loop through every page in the PDF
        for page_num in range(len(doc)):
            page = doc.load_page(page_num) # Load the current page
            text += page.get_text()        # Extract text and append it
            
        return text
    except Exception as e:
        return f"Error reading PDF: {e}"

# 2. Execute the function and store the result
raw_cv_text = extract_text_from_pdf(pdf_path)

# 3. Print the result
print(f"=== EXTRACTED CV TEXT FROM {pdf_path} ===")
print(raw_cv_text[:1000]) # Slicing the first 1000 characters

=== EXTRACTED CV TEXT FROM CVs/ResumeCyberSecurityAnalystBasic.pdf ===
RAJA SHEKAR REDDY SEELAM
sraja456@outlook.com · +1(786)620-5920 · linkedin.com/in/raja045 · github.com/raja045 · rajareddy.site
EDUCATION
Masters in Cyber Security | CGPA: 3.8/4.0
Jan 2024 – Dec 2025
Florida International University; Miami, Florida
• Related CourseWork: Software Vulnerabilities and Security, Advanced Digital Forensics, Cryptography and Network Security
PROJECTS
HomeLab-Security-Operations-Center
May 2024 - June 2024
• Architected and deployed a SOC lab with end-to-end threat detection and response, gaining expertise in SOC architecture,
network segmentation, and blue team workﬂows.
• Conﬁgured & optimized Wazuh SIEM for real-time alert monitoring and correlation, processing 10,000+ events, enhancing
threat detection and streamlining log analysis, while reducing false positives by 30%.
• Deployed and integrated OPNSense, Wireguard, and Suricata across Linux and Windows, increasing monitoring coverage

In [2]:
import re

def chunk_cv_robust_headers(text, user_id):
    """
    Chunks CV text by looking for exact UPPERCASE headers.
    Uses Positive Lookahead to check if the header is followed by a newline (\n) or a colon (:).
    """
    # FIXED: Added the missing commas at the end of the lines to prevent implicit string concatenation.
    headers = [
        'EDUCATION', 'EDUCATIONAL', 'EXPERIENCE', 'WORK EXPERIENCE', 'WORK', 
        'PROFESSIONAL EXPERIENCE', 'LEADERSHIP EXPERIENCE', 'RELEVANT EXPERIENCE',
        'PROJECTS', 'PERSONAL PROJECTS', 'ACADEMIC PROJECTS', 'NOTABLE', 
        'STUDENT ORGANIZATION', 'ORGANIZATION', 'ORGANIZATIONAL',
        'SKILLS', 'TECHNICAL SKILLS', 'CERTIFICATIONS', 'EXTRACURRICULAR',
        'SUMMARY', 'PROFESSIONAL SUMMARY', 'OBJECTIVE', 'PUBLICATIONS', 'INTERESTS', 'LANGUAGES'
    ]
    
    # (?=[ \t]*(?:\n|:|$)) ensures the matched word is followed by space/tab, then newline/colon/end-of-file
    pattern = r'\b(' + '|'.join(headers) + r')\b(?=[ \t]*(?:\n|:|$))'
    
    parts = re.split(pattern, text)
    
    chunks = []
    
    if parts[0].strip():
        chunks.append({
            "metadata": {
                "user_id": user_id,
                "section": "Contact & Summary",
                "text": parts[0].strip()
            }
        })
        
    for i in range(1, len(parts), 2):
        header_name = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        
        if content:
            chunks.append({
                "metadata": {
                    "user_id": user_id,
                    "section": header_name,
                    "text": content
                }
            })
            
    return chunks

# --- Test the Function ---
dummy_user_id = "usr_raja_001"
cv_chunks_final = chunk_cv_robust_headers(raw_cv_text, dummy_user_id)

print(f"✅ CV successfully chunked into {len(cv_chunks_final)} sections!\n")

for i, chunk in enumerate(cv_chunks_final):
    meta = chunk["metadata"]
    print(f"--- Chunk {i+1} | Section: {meta['section']} ---")
    print(f"{meta['text'][:150]}...\n")

✅ CV successfully chunked into 6 sections!

--- Chunk 1 | Section: Contact & Summary ---
RAJA SHEKAR REDDY SEELAM
sraja456@outlook.com · +1(786)620-5920 · linkedin.com/in/raja045 · github.com/raja045 · rajareddy.site...

--- Chunk 2 | Section: EDUCATION ---
Masters in Cyber Security | CGPA: 3.8/4.0
Jan 2024 – Dec 2025
Florida International University; Miami, Florida
• Related CourseWork: Software Vulnerab...

--- Chunk 3 | Section: PROJECTS ---
HomeLab-Security-Operations-Center
May 2024 - June 2024
• Architected and deployed a SOC lab with end-to-end threat detection and response, gaining ex...

--- Chunk 4 | Section: PROFESSIONAL EXPERIENCE ---
Florida International University | Miami, Florida
Feb 2024 – Present
Graduate Research & Student Assistant
• Collected data via web scraping using Pyt...

--- Chunk 5 | Section: SKILLS ---
Security Operations (SOC) & SIEM: Splunk, Wazuh, Alert Triage, Incident Response, Antivirus, Log Analysis, EDR Solutions, Forensics.
Network Security:...

-

In [3]:
import sys
import os
from dotenv import load_dotenv
from pinecone import Pinecone

# 1. Path Setup: Point to the main server directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# 2. LOAD ENV
load_dotenv("../.env")

# 3. IMPORT EMBEDDING FUNCTION
from app.services.embedding import embed_query

# 4. Initialize Pinecone
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = os.environ.get("PINECONE_INDEX_NAME")
index = pc.Index(index_name)

# 5. Upsert Function
def process_and_upsert_cv(chunks_list, user_id):
    vectors_to_upsert = []
    print(f"Processing {len(chunks_list)} chunks for user: {user_id}...\n")
    
    for idx, chunk in enumerate(chunks_list):
        text_content = chunk['metadata']['text']
        section_name = chunk['metadata']['section']
        
        if not text_content.strip():
            continue
            
        try:
            # Using embed_query function
            embedding_values = embed_query(text_content)
            
            chunk_id = f"{user_id}-chunk-{idx}"
            vector_data = {
                "id": chunk_id,
                "values": embedding_values,
                "metadata": {
                    "user_id": user_id,
                    "section": section_name,
                    "text": text_content 
                }
            }
            vectors_to_upsert.append(vector_data)
            print(f"✅ Successfully embedded: Chunk {idx+1} | {section_name}")
            
        except Exception as e:
            print(f"❌ Error embedding Chunk {idx+1}: {e}")

    # Send data to Pinecone
    if vectors_to_upsert:
        print(f"\nUploading to Pinecone index: '{index_name}'...")
        index.upsert(vectors=vectors_to_upsert)
        print("🚀 Upsert Complete! Integrated perfectly with Embedding code.")
    else:
        print("\n⚠️ No vectors to upsert.")

# 6. Execute the function using the finalized CV chunks
process_and_upsert_cv(cv_chunks_final, dummy_user_id)

Loading SentenceTransformer model (all-mpnet-base-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Processing 6 chunks for user: usr_raja_001...

✅ Successfully embedded: Chunk 1 | Contact & Summary
✅ Successfully embedded: Chunk 2 | EDUCATION
✅ Successfully embedded: Chunk 3 | PROJECTS
✅ Successfully embedded: Chunk 4 | PROFESSIONAL EXPERIENCE
✅ Successfully embedded: Chunk 5 | SKILLS
✅ Successfully embedded: Chunk 6 | CERTIFICATIONS

Uploading to Pinecone index: 'knowledge-base-1'...
🚀 Upsert Complete! Integrated perfectly with Embedding code.
